In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import json, base64
from IPython.display import display, HTML

In [ ]:
CLASS_NAMES = [
    'Áo thun (T-shirt)', 'Quần dài (Trouser)', 'Áo len (Pullover)',
    'Váy (Dress)', 'Áo khoác (Coat)', 'Dép (Sandal)',
    'Áo sơ mi (Shirt)', 'Giày thể thao (Sneaker)',
    'Túi xách (Bag)', 'Bốt cổ cao (Ankle boot)'
]

CLASS_EMOJI = ['👕','👖','🧥','👗','🧥','👡','👔','👟','👜','👢']

(x_train, y_train), (x_test, y_test) = keras.datasets.fashion_mnist.load_data()
x_train = x_train.astype('float32') / 255.0
x_test  = x_test.astype('float32')  / 255.0
x_train_flat = x_train.reshape(-1, 784)
x_test_flat  = x_test.reshape(-1, 784)

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(x_train[i], cmap='gray')
    ax.set_title(f'{CLASS_EMOJI[y_train[i]]} {CLASS_NAMES[y_train[i]][:12]}', fontsize=9)
    ax.axis('off')
plt.suptitle('Mẫu dữ liệu Fashion-MNIST', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
def build_ann_model():
    model = keras.Sequential([
        layers.Input(shape=(784,)),
        layers.Dense(512, activation='relu', name='hidden_1'),
        layers.BatchNormalization(),
        layers.Dropout(0.3),
        layers.Dense(256, activation='relu', name='hidden_2'),
        layers.BatchNormalization(),
        layers.Dropout(0.3),
        layers.Dense(128, activation='relu', name='hidden_3'),
        layers.BatchNormalization(),
        layers.Dropout(0.2),
        layers.Dense(10, activation='softmax', name='output')
    ], name='FashionANN')

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

model = build_ann_model()
model.summary()

In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_accuracy', patience=5, restore_best_weights=True
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6
    )
]
history = model.fit(
    x_train_flat, y_train,
    epochs=30,
    batch_size=256,
    validation_split=0.15,
    callbacks=callbacks,
    verbose=1
)
test_loss, test_acc = model.evaluate(x_test_flat, y_test, verbose=0)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history.history['accuracy'], label='Train', color='#2196F3', linewidth=2)
ax1.plot(history.history['val_accuracy'], label='Validation', color='#FF5722', linewidth=2)
ax1.set_title('Accuracy theo Epoch', fontsize=14, fontweight='bold')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Accuracy')
ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.plot(history.history['loss'], label='Train', color='#2196F3', linewidth=2)
ax2.plot(history.history['val_loss'], label='Validation', color='#FF5722', linewidth=2)
ax2.set_title('Loss theo Epoch', fontsize=14, fontweight='bold')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Loss')
ax2.legend(); ax2.grid(True, alpha=0.3)

plt.suptitle(f'Kết quả huấn luyện | Test Acc: {test_acc*100:.1f}%',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
import json

def export_model_weights(model):
    """Xuất tất cả weights và biases của Dense layers ra dict"""
    layers_data = []
    for layer in model.layers:
        if isinstance(layer, keras.layers.Dense):
            W, b = layer.get_weights()
            layers_data.append({
                'name': layer.name,
                'weights': W.tolist(),
                'biases': b.tolist(),
                'activation': layer.activation.__name__
            })
    return layers_data

model_data = export_model_weights(model)
model_json = json.dumps(model_data)

for ld in model_data:
    W = np.array(ld['weights'])

In [ ]:
html_app = f"""
<style>
  @import url('https://fonts.googleapis.com/css2?family=Space+Grotesk:wght@400;500;600;700&display=swap');
  
  #fashion-app {{
    font-family: 'Space Grotesk', sans-serif;
    background: linear-gradient(135deg, #0f0f1a 0%, #1a1a2e 50%, #16213e 100%);
    min-height: 100vh;
    padding: 20px;
    color: #e0e0ff;
  }}
  
  .app-header {{
    text-align: center;
    margin-bottom: 24px;
  }}
  
  .app-header h1 {{
    font-size: 28px;
    font-weight: 700;
    background: linear-gradient(90deg, #a78bfa, #60a5fa, #34d399);
    -webkit-background-clip: text;
    -webkit-text-fill-color: transparent;
    margin: 0 0 6px;
  }}
  
  .app-header p {{
    color: #8888aa;
    font-size: 14px;
    margin: 0;
  }}
  
  .main-layout {{
    display: flex;
    gap: 20px;
    align-items: flex-start;
    max-width: 960px;
    margin: 0 auto;
  }}
  
  .canvas-section {{
    flex: 0 0 auto;
  }}
  
  .canvas-wrapper {{
    background: #0a0a14;
    border: 2px solid #3333aa;
    border-radius: 16px;
    padding: 16px;
    box-shadow: 0 0 40px rgba(100,100,255,0.15);
  }}
  
  #drawCanvas {{
    display: block;
    border-radius: 8px;
    cursor: crosshair;
    background: #000;
  }}
  
  .canvas-controls {{
    display: flex;
    gap: 10px;
    margin-top: 12px;
    justify-content: center;
  }}
  
  .btn {{
    padding: 10px 20px;
    border: none;
    border-radius: 8px;
    font-family: inherit;
    font-size: 14px;
    font-weight: 600;
    cursor: pointer;
    transition: all 0.2s;
  }}
  
  .btn-clear {{
    background: #2a2a4a;
    color: #a0a0cc;
    border: 1px solid #4444aa;
  }}
  
  .btn-clear:hover {{ background: #3a3a5a; }}
  
  .btn-predict {{
    background: linear-gradient(135deg, #6366f1, #4f46e5);
    color: white;
    box-shadow: 0 4px 15px rgba(99,102,241,0.4);
  }}
  
  .btn-predict:hover {{ transform: translateY(-1px); box-shadow: 0 6px 20px rgba(99,102,241,0.5); }}
  
  .result-section {{
    flex: 1;
  }}
  
  .prediction-card {{
    background: #111128;
    border: 1px solid #2a2a5a;
    border-radius: 16px;
    padding: 20px;
    margin-bottom: 16px;
    min-height: 120px;
    display: flex;
    align-items: center;
    justify-content: center;
  }}
  
  .result-main {{
    text-align: center;
  }}
  
  .result-emoji {{
    font-size: 56px;
    margin-bottom: 8px;
    filter: drop-shadow(0 0 12px rgba(167,139,250,0.6));
  }}
  
  .result-name {{
    font-size: 22px;
    font-weight: 700;
    color: #c4b5fd;
    margin-bottom: 6px;
  }}
  
  .result-confidence {{
    font-size: 14px;
    color: #6666aa;
  }}
  
  .confidence-fill {{
    display: inline-block;
    padding: 2px 10px;
    background: rgba(99,102,241,0.2);
    border: 1px solid rgba(99,102,241,0.4);
    border-radius: 20px;
    color: #a78bfa;
    font-weight: 600;
    margin-top: 4px;
  }}
  
  .placeholder-text {{
    color: #3a3a5a;
    font-size: 14px;
    text-align: center;
  }}
  
  .bars-container {{
    background: #111128;
    border: 1px solid #2a2a5a;
    border-radius: 16px;
    padding: 16px;
  }}
  
  .bars-title {{
    font-size: 12px;
    font-weight: 600;
    text-transform: uppercase;
    letter-spacing: 0.08em;
    color: #5555aa;
    margin-bottom: 12px;
  }}
  
  .bar-row {{
    display: flex;
    align-items: center;
    gap: 8px;
    margin-bottom: 7px;
  }}
  
  .bar-label {{
    font-size: 11px;
    color: #6666aa;
    width: 130px;
    text-align: right;
    flex-shrink: 0;
    white-space: nowrap;
    overflow: hidden;
    text-overflow: ellipsis;
  }}
  
  .bar-track {{
    flex: 1;
    height: 14px;
    background: #1a1a30;
    border-radius: 7px;
    overflow: hidden;
  }}
  
  .bar-fill {{
    height: 100%;
    border-radius: 7px;
    background: linear-gradient(90deg, #4f46e5, #a78bfa);
    transition: width 0.5s cubic-bezier(0.4,0,0.2,1);
  }}
  
  .bar-fill.top {{
    background: linear-gradient(90deg, #059669, #34d399);
  }}
  
  .bar-pct {{
    font-size: 11px;
    color: #5555aa;
    width: 36px;
    text-align: right;
    flex-shrink: 0;
  }}
  
  .status-indicator {{
    display: flex;
    align-items: center;
    gap: 6px;
    font-size: 12px;
    color: #34d399;
    margin-top: 10px;
    justify-content: center;
  }}
  
  .status-dot {{
    width: 7px;
    height: 7px;
    background: #34d399;
    border-radius: 50%;
    animation: pulse 1.5s infinite;
  }}
  
  @keyframes pulse {{
    0%, 100% {{ opacity: 1; transform: scale(1); }}
    50% {{ opacity: 0.5; transform: scale(0.8); }}
  }}
  
  .brush-controls {{
    display: flex;
    align-items: center;
    gap: 10px;
    margin-top: 10px;
    justify-content: center;
    font-size: 13px;
    color: #6666aa;
  }}
  
  .brush-controls input {{
    width: 80px;
    accent-color: #6366f1;
  }}
  
  .preview-canvas-wrap {{
    text-align: center;
    margin-top: 12px;
  }}
  
  .preview-label {{
    font-size: 11px;
    color: #3a3a5a;
    margin-bottom: 4px;
  }}
  
  #previewCanvas {{
    border: 1px solid #2a2a5a;
    border-radius: 4px;
    image-rendering: pixelated;
  }}
</style>

<div id="fashion-app">
  <div class="app-header">
    <h1>👗 Fashion AI — Nhận dạng quần áo</h1>
    <p>Vẽ quần áo bằng chuột · AI nhận dạng realtime · ANN 3 hidden layers</p>
  </div>
  
  <div class="main-layout">
    <div class="canvas-section">
      <div class="canvas-wrapper">
        <canvas id="drawCanvas" width="280" height="280"></canvas>
        
        <div class="brush-controls">
          <span>Bút:</span>
          <input type="range" id="brushSize" min="8" max="30" value="16">
          <span id="brushLabel">16px</span>
        </div>
        
        <div class="canvas-controls">
          <button class="btn btn-clear" onclick="clearCanvas()">🗑 Xóa</button>
          <button class="btn btn-predict" onclick="predictNow()">🔍 Nhận dạng</button>
        </div>
        
        <div class="preview-canvas-wrap">
          <div class="preview-label">Input 28×28 (gửi vào model)</div>
          <canvas id="previewCanvas" width="56" height="56"
                  style="width:56px;height:56px;"></canvas>
        </div>
      </div>
    </div>
    
    <div class="result-section">
      <div class="prediction-card" id="predictionCard">
        <p class="placeholder-text">✏️ Vẽ một món đồ thời trang rồi nhấn <strong>Nhận dạng</strong></p>
      </div>
      
      <div class="bars-container">
        <div class="bars-title">Xác suất từng loại</div>
        <div id="barsArea">
          {chr(10).join(f'<div class="bar-row" id="bar_{i}"><span class="bar-label">{CLASS_EMOJI[i]} {CLASS_NAMES[i][:18]}</span><div class="bar-track"><div class="bar-fill" id="fill_{i}" style="width:0%"></div></div><span class="bar-pct" id="pct_{i}">0%</span></div>' for i in range(10))}
        </div>
      </div>
      
      <div class="status-indicator">
        <div class="status-dot"></div>
      </div>
    </div>
  </div>
</div>

<script>
// ---- Model weights (loaded from Python) ----
const MODEL_DATA = {model_json};

const CLASS_NAMES = {json.dumps(CLASS_NAMES)};
const CLASS_EMOJI = {json.dumps(CLASS_EMOJI)};

// ---- Canvas setup ----
const canvas = document.getElementById('drawCanvas');
const ctx = canvas.getContext('2d');
const preview = document.getElementById('previewCanvas');
const pctx = preview.getContext('2d');

let painting = false;
let brushSize = 16;
let autoTimer = null;

ctx.fillStyle = '#000';
ctx.fillRect(0, 0, 280, 280);

document.getElementById('brushSize').addEventListener('input', e => {{
  brushSize = +e.target.value;
  document.getElementById('brushLabel').textContent = brushSize + 'px';
}});

function getPos(e) {{
  const r = canvas.getBoundingClientRect();
  if (e.touches) {{
    return {{ x: e.touches[0].clientX - r.left, y: e.touches[0].clientY - r.top }};
  }}
  return {{ x: e.clientX - r.left, y: e.clientY - r.top }};
}}

canvas.addEventListener('mousedown',  e => {{ painting = true; draw(e); }});
canvas.addEventListener('mousemove',  e => {{ if(painting) draw(e); }});
canvas.addEventListener('mouseup',    () => stopDraw());
canvas.addEventListener('mouseleave', () => stopDraw());
canvas.addEventListener('touchstart', e => {{ e.preventDefault(); painting=true; draw(e); }}, {{passive:false}});
canvas.addEventListener('touchmove',  e => {{ e.preventDefault(); if(painting) draw(e); }}, {{passive:false}});
canvas.addEventListener('touchend',   () => stopDraw());

function draw(e) {{
  const pos = getPos(e);
  ctx.beginPath();
  ctx.arc(pos.x, pos.y, brushSize/2, 0, Math.PI*2);
  ctx.fillStyle = 'white';
  ctx.fill();
  scheduleAutoPredict();
}}

function stopDraw() {{
  painting = false;
}}

function clearCanvas() {{
  ctx.fillStyle = '#000';
  ctx.fillRect(0, 0, 280, 280);
  pctx.fillStyle = '#000';
  pctx.fillRect(0, 0, 56, 56);
  resetBars();
  document.getElementById('predictionCard').innerHTML =
    '<p class="placeholder-text">✏️ Vẽ một món đồ thời trang rồi nhấn <strong>Nhận dạng</strong></p>';
}}

function scheduleAutoPredict() {{
  clearTimeout(autoTimer);
  autoTimer = setTimeout(predictNow, 600);
}}

// ---- Preprocess: 280x280 → 28x28 float array ----
function getInputArray() {{
  // Scale down to 28x28
  const tmp = document.createElement('canvas');
  tmp.width = 28; tmp.height = 28;
  const tc = tmp.getContext('2d');
  tc.drawImage(canvas, 0, 0, 280, 280, 0, 0, 28, 28);

  // Show preview (scaled to 56x56)
  pctx.clearRect(0, 0, 56, 56);
  pctx.drawImage(tmp, 0, 0, 56, 56);

  // Extract pixels & normalize
  const imgData = tc.getImageData(0, 0, 28, 28).data;
  const input = new Float32Array(784);
  for (let i = 0; i < 784; i++) {{
    // Grayscale = average of RGB
    input[i] = (imgData[i*4] + imgData[i*4+1] + imgData[i*4+2]) / (3 * 255.0);
  }}
  return input;
}}

// ---- ANN Forward Pass (pure JS) ----
function relu(x) {{ return Math.max(0, x); }}

function softmax(arr) {{
  const max = Math.max(...arr);
  const exps = arr.map(x => Math.exp(x - max));
  const sum = exps.reduce((a,b) => a+b, 0);
  return exps.map(x => x/sum);
}}

function denseLayer(input, weights, biases, activation) {{
  const inSize = input.length;
  const outSize = biases.length;
  const output = new Float32Array(outSize);
  for (let j = 0; j < outSize; j++) {{
    let sum = biases[j];
    for (let i = 0; i < inSize; i++) {{
      sum += input[i] * weights[i][j];
    }}
    output[j] = activation === 'relu' ? relu(sum) : sum;
  }}
  return output;
}}

function forwardPass(input) {{
  let x = input;
  for (const layer of MODEL_DATA) {{
    x = denseLayer(x, layer.weights, layer.biases, layer.activation);
  }}
  return softmax(Array.from(x));
}}

// ---- Predict ----
function predictNow() {{
  const input = getInputArray();
  const hasContent = input.some(v => v > 0.01);
  if (!hasContent) return;

  const probs = forwardPass(input);
  const topIdx = probs.indexOf(Math.max(...probs));
  const topProb = probs[topIdx];

  // Update main card
  document.getElementById('predictionCard').innerHTML = `
    <div class="result-main">
      <div class="result-emoji">${{CLASS_EMOJI[topIdx]}}</div>
      <div class="result-name">${{CLASS_NAMES[topIdx]}}</div>
      <div class="result-confidence">
        Độ tin cậy
        <div class="confidence-fill">${{(topProb*100).toFixed(1)}}%</div>
      </div>
    </div>
  `;

  // Update bars
  for (let i = 0; i < 10; i++) {{
    const pct = (probs[i]*100).toFixed(1);
    const fill = document.getElementById('fill_' + i);
    const pctEl = document.getElementById('pct_' + i);
    fill.style.width = pct + '%';
    fill.className = 'bar-fill' + (i === topIdx ? ' top' : '');
    pctEl.textContent = pct + '%';
  }}
}}

function resetBars() {{
  for (let i = 0; i < 10; i++) {{
    document.getElementById('fill_' + i).style.width = '0%';
    document.getElementById('pct_' + i).textContent = '0%';
  }}
}}

console.log('Fashion AI App loaded! Model layers:', MODEL_DATA.length);
</script>
"""

display(HTML(html_app))

In [ ]:
n = 15
indices = np.random.choice(len(x_test), n, replace=False)
x_sample = x_test_flat[indices]
y_true   = y_test[indices]
y_pred   = np.argmax(model.predict(x_sample, verbose=0), axis=1)

fig, axes = plt.subplots(3, 5, figsize=(15, 9))
for i, ax in enumerate(axes.flat):
    ax.imshow(x_test[indices[i]], cmap='gray')
    ok = y_pred[i] == y_true[i]
    color = 'green' if ok else 'red'
    ax.set_title(
        f'{"✅" if ok else "❌"} Pred: {CLASS_NAMES[y_pred[i]][:10]}\nTrue: {CLASS_NAMES[y_true[i]][:10]}',
        fontsize=8, color=color
    )
    ax.axis('off')

plt.suptitle('Kiểm tra model trên test set', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

acc = np.mean(y_pred == y_true)